# explanatory data analysis

# imports

In [13]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = duckdb.connect('../../database/financial_data.duckdb', read_only=True)

In [14]:
def query(sql):
    return conn.execute(sql).df()

# rows count 4h only

In [15]:
query("""
SELECT asset_symbol, COUNT(*) as rows, 
       MIN(date) as first_date, MAX(date) as last_date
FROM gold_crypto_features 
WHERE interval = '4h'
GROUP BY asset_symbol 
ORDER BY rows DESC
""")

,asset_symbol,rows,first_date,last_date


# all intervals comparison

In [16]:
query("""
SELECT interval, COUNT(*) as total_rows, 
       COUNT(DISTINCT asset_symbol) as symbols
FROM gold_crypto_features 
GROUP BY interval 
ORDER BY interval
""")

,interval,total_rows,symbols
0,1d,17581,11
1,1h,472124,11
2,W,687,10


# nan check like oi and funding rate for 4h

In [17]:
query("""
SELECT asset_symbol,
       COUNT(*) as total_rows,
       SUM(CASE WHEN open_interest IS NULL THEN 1 ELSE 0 END) as null_oi,
       SUM(CASE WHEN funding_rate IS NULL THEN 1 ELSE 0 END) as null_fr,
       SUM(CASE WHEN rsi_14 IS NULL THEN 1 ELSE 0 END) as null_rsi,
       SUM(CASE WHEN macd IS NULL THEN 1 ELSE 0 END) as null_macd
FROM gold_crypto_features
WHERE interval = '4h'
GROUP BY asset_symbol
ORDER BY total_rows DESC
""")

,asset_symbol,total_rows,null_oi,null_fr,null_rsi,null_macd


In [18]:
query("""
SELECT interval, COUNT(*) as rows
FROM gold_crypto_features
GROUP BY interval
ORDER BY interval
""")


,interval,rows
0,1d,17581
1,1h,472124
2,W,687


In [21]:
query("""
SELECT symbol, interval, date, open, close, volume
FROM clean_bybit_crypto
WHERE interval = '4h'
LIMIT 5
""")


,symbol,interval,date,open,close,volume
0,1000PEPEUSDT,4h,2023-05-03 08:00:00,0.000922,0.000844,1.385406e+09
1,1000PEPEUSDT,4h,2023-05-03 12:00:00,0.000844,0.000950,2.541815e+10
2,1000PEPEUSDT,4h,2023-05-03 16:00:00,0.000950,0.001048,4.118626e+10
3,1000PEPEUSDT,4h,2023-05-03 20:00:00,0.001048,0.001059,4.667071e+10
4,1000PEPEUSDT,4h,2023-05-04 00:00:00,0.001059,0.001166,2.591998e+10


In [20]:
query("""
SELECT * FROM dim_interval ORDER BY interval_id
""")

,interval_id,interval_code,interval_minutes,interval_description
0,1,1h,60,Hourly
1,2,60,60,Hourly (Bybit)
2,3,1d,1440,Daily
3,4,D,1440,Daily (Bybit)
4,5,1wk,10080,Weekly
5,6,W,10080,Weekly (Bybit)
6,7,1mo,43200,Monthly
7,8,M,43200,Monthly (Bybit)


In [22]:
conn.close()